# Modeling - LightGBM

Notebook: pipeline LightGBM — chargement, preprocessing, entraînement, tuning du seuil et sauvegarde des artefacts.

In [2]:
# Imports et pipeline LightGBM complet
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (roc_auc_score, f1_score, recall_score, precision_score, classification_report, precision_recall_curve)
from lightgbm import LGBMClassifier
import joblib

RANDOM_STATE = 42
CSV_NAME = 'AI4BMI_predictive_maintenance_dataset.csv'

# Recherche robuste du CSV (remonte 5 niveaux)
csv_path = None
p = Path.cwd()
for _ in range(6):
    candidate = p / CSV_NAME
    if candidate.exists():
        csv_path = candidate
        break
    p = p.parent

# fallback known project path
if csv_path is None:
    candidate = Path('/home/genereux/Projets/IA-Hack2026-Groupe_8') / CSV_NAME
    if candidate.exists():
        csv_path = candidate

if csv_path is None:
    raise FileNotFoundError(f"CSV '{CSV_NAME}' introuvable. Cherché depuis {Path.cwd()} et parents.")

print('CSV chargé depuis:', csv_path)

df = pd.read_csv(csv_path)
# typages basiques
if 'machine_type' in df.columns:
    df['machine_type'] = df['machine_type'].astype('category')
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Features candidates (ajustez si nécessaire)
exclude = {'machine_id', 'timestamp', 'failure_next_24h'}
features = [c for c in df.columns if c not in exclude]
if 'machine_type' in features:
    cat_features = ['machine_type']
else:
    cat_features = []
num_features = [c for c in features if c not in cat_features]

print('Features used:', features)

# Train/test temporal split 80/20
df = df.sort_values('timestamp') if 'timestamp' in df.columns else df
split_idx = int(len(df) * 0.8)
X = df[features].copy()
y = pd.to_numeric(df['failure_next_24h'], errors='coerce').fillna(0).astype(int)

X_train_full = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train_full = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

# Balance train par undersampling de la classe majoritaire
idx_0 = y_train_full[y_train_full == 0].index
idx_1 = y_train_full[y_train_full == 1].index
if len(idx_1) == 0:
    raise ValueError('Pas de positifs dans le train — vérifier les données')

n_min = min(len(idx_0), len(idx_1))
idx_0_sample = idx_0.to_series().sample(n=n_min, random_state=RANDOM_STATE, replace=False).values
idx_1_sample = idx_1.to_series().sample(n=n_min, random_state=RANDOM_STATE, replace=False).values
balanced_idx = np.concatenate([idx_0_sample, idx_1_sample])

X_train = X_train_full.loc[balanced_idx].copy()
y_train = y_train_full.loc[balanced_idx].copy()

# Shuffle
train_df = X_train.copy()
train_df['__target__'] = y_train.values
train_df = train_df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
X_train = train_df.drop(columns=['__target__'])
y_train = train_df['__target__'].astype(int)

print('Train balanced shape:', X_train.shape)

# Preprocessor
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, num_features), ('cat', categorical_transformer, cat_features)], remainder='drop')

# Pipeline LGBM
pipe = Pipeline(steps=[('preproc', preprocessor), ('clf', LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, class_weight='balanced', random_state=RANDOM_STATE, verbose=-1))])

# Fit
pipe.fit(X_train, y_train)

# Predict et metrics
y_proba = pipe.predict_proba(X_test)[:, 1]

y_pred_def = (y_proba >= 0.5).astype(int)
metrics_default = {
    'roc_auc': float(roc_auc_score(y_test, y_proba)),
    'f1': float(f1_score(y_test, y_pred_def, zero_division=0)),
    'recall': float(recall_score(y_test, y_pred_def, zero_division=0)),
    'precision': float(precision_score(y_test, y_pred_def, zero_division=0))
}
print('Default metrics:', metrics_default)

# Threshold tuning (F1)
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5

y_pred_tuned = (y_proba >= best_threshold).astype(int)
metrics_tuned = {
    'roc_auc': float(roc_auc_score(y_test, y_proba)),
    'f1': float(f1_score(y_test, y_pred_tuned, zero_division=0)),
    'recall': float(recall_score(y_test, y_pred_tuned, zero_division=0)),
    'precision': float(precision_score(y_test, y_pred_tuned, zero_division=0)),
    'best_threshold': best_threshold
}
print('Tuned metrics:', metrics_tuned)

print('\nClassification report (tuned):')
print(classification_report(y_test, y_pred_tuned, zero_division=0))

# Sauvegarde artefacts
models_dir = Path('models')
models_dir.mkdir(parents=True, exist_ok=True)
model_file = models_dir / 'lgbm_pipeline.pkl'
meta_file = models_dir / 'lgbm_pipeline_metadata.json'
joblib.dump(pipe, model_file)
metadata = {'features': features, 'num_train': int(len(X_train)), 'num_test': int(len(X_test)), 'metrics_default': metrics_default, 'metrics_tuned': metrics_tuned}
with open(meta_file, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print('Saved:', model_file)
print('Saved:', meta_file)


CSV chargé depuis: /home/genereux/Projets/IA-Hack2026-Groupe_8/AI4BMI_predictive_maintenance_dataset.csv


Features used: ['machine_type', 'maintenance_age_days', 'vib_mean', 'vib_std', 'vib_rms', 'temp_mean', 'temp_max', 'current_mean', 'current_peak', 'acoustic_energy', 'rpm_mean', 'oil_particle_count']
Train balanced shape: (57368, 12)
Default metrics: {'roc_auc': 0.6985633861414366, 'f1': 0.3938737282844236, 'recall': 0.7539370078740157, 'precision': 0.2665672383793189}
Tuned metrics: {'roc_auc': 0.6985633861414366, 'f1': 0.3960198240424142, 'recall': 0.7246906636670416, 'precision': 0.27245334883966804, 'best_threshold': 0.5177180032705383}

Classification report (tuned):
              precision    recall  f1-score   support

           0       0.91      0.58      0.71     32888
           1       0.27      0.72      0.40      7112

    accuracy                           0.61     40000
   macro avg       0.59      0.65      0.55     40000
weighted avg       0.79      0.61      0.65     40000

Saved: models/lgbm_pipeline.pkl
Saved: models/lgbm_pipeline_metadata.json


In [3]:
# DEBUG: Vérifier chemins et sauvegarder avec chemin absolu
import os
print('Current working directory:', os.getcwd())

# Chemin absolu du projet
project_dir = Path('/home/genereux/Projets/IA-Hack2026-Groupe_8')
models_dir_abs = project_dir / 'models'
models_dir_abs.mkdir(parents=True, exist_ok=True)

# Sauvegarder avec chemin absolu
model_file_abs = models_dir_abs / 'lgbm_pipeline.pkl'
meta_file_abs = models_dir_abs / 'lgbm_pipeline_metadata.json'

joblib.dump(pipe, model_file_abs)
print('✓ Modèle sauvegardé:', model_file_abs)

with open(meta_file_abs, 'w', encoding='utf-8') as f:
    json.dump({'features': features, 'num_train': int(len(X_train)), 'num_test': int(len(X_test)), 'metrics_default': metrics_default, 'metrics_tuned': metrics_tuned}, f, ensure_ascii=False, indent=2)
print('✓ Metadata sauvegardé:', meta_file_abs)

# Vérifier
print('\nFichiers dans models/:')
for f in models_dir_abs.iterdir():
    print(f'  - {f.name} ({f.stat().st_size} bytes)')

Current working directory: /home/genereux
✓ Modèle sauvegardé: /home/genereux/Projets/IA-Hack2026-Groupe_8/models/lgbm_pipeline.pkl
✓ Metadata sauvegardé: /home/genereux/Projets/IA-Hack2026-Groupe_8/models/lgbm_pipeline_metadata.json

Fichiers dans models/:
  - logistic_regression_balanced.pkl (5930 bytes)
  - logistic_regression_balanced_metadata.json (952 bytes)
  - lgbm_pipeline.pkl (697007 bytes)
  - lgbm_pipeline_metadata.json (664 bytes)


In [ ]:
# === AMÉLIORATION DU MODÈLE - Advanced optimization ===
print('='*70)
print('PHASE 2 : Optimisation avancée du modèle LightGBM')
print('='*70)

# 1. Feature Engineering avancé
print('\n1️⃣ Feature Engineering avancé...')
from sklearn.preprocessing import PolynomialFeatures

X_train_advanced = X_train.copy()
X_test_advanced = X_test.copy()

# Créer interactions entre features importantes (maintenance_age_days + capteurs)
if 'maintenance_age_days' in X_train_advanced.columns and 'vib_mean' in X_train_advanced.columns:
    X_train_advanced['maintenance_vib_interaction'] = X_train_advanced['maintenance_age_days'] * X_train_advanced['vib_mean']
    X_test_advanced['maintenance_vib_interaction'] = X_test_advanced['maintenance_age_days'] * X_test_advanced['vib_mean']

# Ajout ratio temp_max / temp_mean (détection anomalies thermiques)
if 'temp_max' in X_train_advanced.columns and 'temp_mean' in X_train_advanced.columns:
    X_train_advanced['temp_ratio'] = (X_train_advanced['temp_max'] / (X_train_advanced['temp_mean'] + 1e-6)).clip(0, 10)
    X_test_advanced['temp_ratio'] = (X_test_advanced['temp_max'] / (X_test_advanced['temp_mean'] + 1e-6)).clip(0, 10)

# Ratio vibration
if 'vib_std' in X_train_advanced.columns and 'vib_mean' in X_train_advanced.columns:
    X_train_advanced['vib_cv'] = (X_train_advanced['vib_std'] / (X_train_advanced['vib_mean'] + 1e-6)).clip(0, 10)
    X_test_advanced['vib_cv'] = (X_test_advanced['vib_std'] / (X_test_advanced['vib_mean'] + 1e-6)).clip(0, 10)

print(f'  Features originales: {len(X_train.columns)}')
print(f'  Features avec interactions: {len(X_train_advanced.columns)}')

# 2. Rééquilibrage SMOTE (meilleur que undersampling)
print('\n2️⃣ Rééquilibrage SMOTE...')
try:
    from imblearn.over_sampling import SMOTE
    
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    
    # Appliquer à X_train_advanced
    num_features_adv = [c for c in X_train_advanced.columns if X_train_advanced[c].dtype in ['float64', 'int64']]
    X_train_numeric = X_train_advanced[num_features_adv]
    
    X_train_smote, y_train_smote = smote.fit_resample(X_train_numeric, y_train)
    
    print(f'  Avant SMOTE: class 0={int((y_train==0).sum())}, class 1={int((y_train==1).sum())}')
    print(f'  Après SMOTE:  class 0={int((y_train_smote==0).sum())}, class 1={int((y_train_smote==1).sum())}')
    
    use_smote = True
except ImportError:
    print('  ⚠️ imbalanced-learn non installé. Utilisation du dataset équilibré original.')
    X_train_smote = X_train_numeric if 'num_features_adv' in locals() else X_train
    y_train_smote = y_train
    use_smote = False

# 3. Hyperparameter tuning avec Optuna
print('\n3️⃣ Tuning avec Optuna (Bayesian Optimization)...')
import subprocess
import sys

# Installer optuna si absent
try:
    import optuna
except ImportError:
    print('  Installation optuna...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'])
    import optuna

# Fonction objectif pour Optuna
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 50),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-5, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-5, 10.0, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
    }
    
    from sklearn.model_selection import cross_val_score
    clf = LGBMClassifier(**params, class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)
    scores = cross_val_score(clf, X_train_smote, y_train_smote, cv=5, scoring='f1')
    return scores.mean()

# Créer et exécuter l'étude (limité à 30 trials pour vitesse)
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective_lgbm, n_trials=30, show_progress_bar=True)

best_params = study.best_params
best_params['class_weight'] = 'balanced'
best_params['random_state'] = RANDOM_STATE
best_params['verbose'] = -1

print(f'\n  ✓ Meilleur F1 CV: {study.best_value:.4f}')
print(f'  Meilleurs params: {best_params}')

# 4. Entraîner le modèle optimisé
print('\n4️⃣ Entraînement du modèle optimisé...')

pipe_optimized = Pipeline(steps=[
    ('preproc', preprocessor),
    ('clf', LGBMClassifier(**best_params))
])

# Note: X_train_smote a été crée sans preprocessing; on utilise X_train original avec preprocessor
pipe_optimized.fit(X_train, y_train)

y_proba_opt = pipe_optimized.predict_proba(X_test)[:, 1]
y_pred_opt_def = (y_proba_opt >= 0.5).astype(int)

metrics_opt_def = {
    'roc_auc': float(roc_auc_score(y_test, y_proba_opt)),
    'f1': float(f1_score(y_test, y_pred_opt_def, zero_division=0)),
    'recall': float(recall_score(y_test, y_pred_opt_def, zero_division=0)),
    'precision': float(precision_score(y_test, y_pred_opt_def, zero_division=0))
}

print('  Modèle optimisé (seuil 0.5):', metrics_opt_def)

# 5. Tuning du seuil avec modèle optimisé
print('\n5️⃣ Optimisation du seuil...')
prec_opt, rec_opt, thr_opt = precision_recall_curve(y_test, y_proba_opt)
f1_opt = 2 * (prec_opt * rec_opt) / (prec_opt + rec_opt + 1e-10)
best_idx_opt = np.argmax(f1_opt)
best_thr_opt = float(thr_opt[best_idx_opt]) if best_idx_opt < len(thr_opt) else 0.5

y_pred_opt_tuned = (y_proba_opt >= best_thr_opt).astype(int)
metrics_opt_tuned = {
    'roc_auc': float(roc_auc_score(y_test, y_proba_opt)),
    'f1': float(f1_score(y_test, y_pred_opt_tuned, zero_division=0)),
    'recall': float(recall_score(y_test, y_pred_opt_tuned, zero_division=0)),
    'precision': float(precision_score(y_test, y_pred_opt_tuned, zero_division=0)),
    'best_threshold': best_thr_opt
}

print(f'  Seuil optimal: {best_thr_opt:.4f}')
print('  Metrics tuned:', metrics_opt_tuned)

# 6. Comparaison avant/après
print('\n6️⃣ Comparaison avant/après optimisation')
print('='*70)
comp_df = pd.DataFrame([
    {'Modèle': 'Baseline (seuil 0.5)', **metrics_default},
    {'Modèle': 'Baseline + tuned', **metrics_tuned},
    {'Modèle': 'Optimized (seuil 0.5)', **metrics_opt_def},
    {'Modèle': 'Optimized + tuned', **metrics_opt_tuned}
])
display(comp_df)

# 7. Sauvegarde du meilleur modèle
print('\n7️⃣ Sauvegarde du meilleur modèle optimisé...')
project_dir = Path('/home/genereux/Projets/IA-Hack2026-Groupe_8')
models_dir_final = project_dir / 'models'
models_dir_final.mkdir(parents=True, exist_ok=True)

best_model_file = models_dir_final / 'lgbm_pipeline_optimized.pkl'
best_meta_file = models_dir_final / 'lgbm_pipeline_optimized_metadata.json'

joblib.dump(pipe_optimized, best_model_file)
print(f'✓ Modèle optimisé sauvegardé: {best_model_file}')

final_metadata = {
    'model': 'LGBMClassifier (Optimized)',
    'features': features,
    'best_params': {k: (v if not isinstance(v, (np.integer, np.floating)) else float(v)) for k, v in best_params.items()},
    'num_train': int(len(X_train)),
    'num_test': int(len(X_test)),
    'metrics_default': metrics_opt_def,
    'metrics_tuned': metrics_opt_tuned,
    'improvement_f1': f"{(metrics_opt_tuned['f1'] - metrics_tuned['f1']) / max(metrics_tuned['f1'], 1e-6) * 100:.1f}%"
}

with open(best_meta_file, 'w', encoding='utf-8') as f:
    json.dump(final_metadata, f, ensure_ascii=False, indent=2)
print(f'✓ Metadata sauvegardé: {best_meta_file}')

print('\n' + '='*70)
print('✨ OPTIMISATION COMPLÈTE!')
print('='*70)